# Generate QA Pairs from ASR Eval CSV using Local Qwen

This notebook is for parallel QA-pair generation using your teammate's file:

`niletts_asr_eval_results.csv`

Expected columns:

- `audio_file`
- `speaker_name`
- `reference_text`
- `asr_text`
- `wer`
- `cer`

Important methodology:

- Generate QA pairs from `reference_text` by default.
- Keep `asr_text`, `wer`, and `cer` in the output for later ASR-context QA evaluation.
- Do **not** generate gold QA answers from `asr_text` unless you only want a noisy auxiliary dataset.
- For the final ASR-error analysis, use the same questions/gold answers generated from `reference_text`, then evaluate once on `context_ref` and once on `context_asr`.

### =====
This is an alternative QA-generation workflow using:

`1_ASR/niletts_asr_eval_finetuning_transcriptions_metrics.csv`

No saved outputs establish this notebook as the source of the final QA files.
Paths and chunk-resume handling were updated during repository cleanup.
### =====


In [ ]:
# =====
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "requirements-qa.txt").is_file():
        requirements_file = candidate / "requirements-qa.txt"
        break
else:
    raise FileNotFoundError("Could not find requirements-qa.txt")

%pip install -r "{requirements_file}"
# =====


In [ ]:
import os
import re
import json
import time
import random
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
device


## 1. Load ASR eval CSV

Upload `niletts_asr_eval_results.csv` to Colab files or place it in your working directory.

This CSV should contain the 950 official eval/test rows.


In [ ]:
# =====
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (
        (candidate / "1_ASR").is_dir()
        and (candidate / "2_QA_GENERATION").is_dir()
    ):
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Open the notebook from inside the project folder."
    )

QA_DIR = PROJECT_ROOT / "2_QA_GENERATION"

# CSV_PATH = "niletts_asr_eval_results.csv"
CSV_PATH = (
    PROJECT_ROOT / "1_ASR"
    / "niletts_asr_eval_finetuning_transcriptions_metrics.csv"
)

OUTPUT_DIR = QA_DIR / "3_test_data"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# if not Path(CSV_PATH).exists():
#     raise FileNotFoundError(
#         f"Could not find {CSV_PATH}. Upload it to Colab or change CSV_PATH to the correct location."
#     )

# asr_eval_df = pd.read_csv(CSV_PATH)

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"Missing ASR results file: {CSV_PATH}")
# =====

required_cols = ["audio_file", "speaker_name", "reference_text", "asr_text", "wer", "cer"]
missing = [c for c in required_cols if c not in asr_eval_df.columns]

if missing:
    raise ValueError(f"Missing columns in ASR CSV: {missing}")

asr_eval_df = asr_eval_df[required_cols].copy()

for col in ["audio_file", "speaker_name", "reference_text", "asr_text"]:
    asr_eval_df[col] = asr_eval_df[col].astype(str).str.strip()

asr_eval_df["wer"] = pd.to_numeric(asr_eval_df["wer"], errors="coerce")
asr_eval_df["cer"] = pd.to_numeric(asr_eval_df["cer"], errors="coerce")

print("ASR eval CSV shape:", asr_eval_df.shape)
print("Mean WER:", asr_eval_df["wer"].mean())
print("Mean CER:", asr_eval_df["cer"].mean())

display(asr_eval_df.head())


In [ ]:
def extract_domain(audio_file):
    name = os.path.basename(str(audio_file)).lower()
    if name.startswith("sales"):
        return "sales"
    if name.startswith("medical"):
        return "medical"
    if name.startswith("general"):
        return "general"
    return "unknown"


asr_eval_df["domain"] = asr_eval_df["audio_file"].apply(extract_domain)
asr_eval_df["split"] = "test"

print("Domain distribution:")
display(asr_eval_df["domain"].value_counts())

print("Speaker distribution:")
display(asr_eval_df["speaker_name"].value_counts())


## 2. Parallel chunk settings

Use this notebook to split the 950 rows between teammates.

Example:

- Person A: `CHUNK_START = 0`, `CHUNK_END = 475`, `RUN_NAME = "test_part_1"`
- Person B: `CHUNK_START = 475`, `CHUNK_END = 950`, `RUN_NAME = "test_part_2"`

Do not let two people generate the same rows unless you intentionally want duplicates.


In [ ]:
# Change these for parallel work.
CHUNK_START = 0
CHUNK_END = 950
RUN_NAME = "test_full"

# Use "reference_text" for correct final methodology.
# Use "asr_text" only if you intentionally want noisy QA pairs, not the final gold test set.
GENERATE_FROM_COLUMN = "reference_text"

# =====
# test_source = asr_eval_df.iloc[CHUNK_START:CHUNK_END].copy().reset_index(drop=True)

if not (0 <= CHUNK_START < CHUNK_END <= len(asr_eval_df)):
    raise ValueError(
        "Chunk bounds must satisfy: "
        "0 <= CHUNK_START < CHUNK_END <= number of ASR rows."
    )

test_source = (asr_eval_df.iloc[CHUNK_START:CHUNK_END].copy().reset_index(drop=True))
# =====

print("Chunk:", CHUNK_START, "to", CHUNK_END)
print("Rows in this chunk:", len(test_source))
print("Generating QA from:", GENERATE_FROM_COLUMN)

display(test_source.head())


## 3. Load local Qwen LLM

This avoids Gemini quota limits. On Colab T4, start with:

`Qwen/Qwen2.5-1.5B-Instruct`

If you have more GPU memory, you may try a larger instruct model, but 1.5B is safer.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LOCAL_LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(LOCAL_LLM_NAME)

local_llm = AutoModelForCausalLM.from_pretrained(
    LOCAL_LLM_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

local_llm.eval()

print("Loaded:", LOCAL_LLM_NAME)


## 4. Prompt and parsing functions

The LLM is asked to return JSON only.

Each answer should be copied from the transcript so it can be checked automatically.


In [ ]:
def normalize_spaces(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def find_answer_start_soft(context, answer):
    context = str(context)
    answer = str(answer)

    exact = context.find(answer)
    if exact != -1:
        return exact

    context_norm = normalize_spaces(context)
    answer_norm = normalize_spaces(answer)

    norm_pos = context_norm.find(answer_norm)
    if norm_pos != -1:
        return norm_pos

    return -1


def build_qa_generation_prompt(context, domain, speaker_name, n_pairs=3):
    return f'''
You are creating a high-quality Arabic question-answering dataset from an Egyptian Arabic transcript.

Transcript domain: {domain}
Speaker: {speaker_name}

Transcript:
"""{context}"""

Generate {n_pairs} useful QA pairs.

Rules:
1. Questions must be in Arabic.
2. Questions must be answerable ONLY from the transcript.
3. The answer must be a short span copied EXACTLY from the transcript.
4. Do not write answers that are not literally present in the transcript.
5. Avoid yes/no questions.
6. Avoid vague questions such as "what is mentioned?"
7. Use varied question types:
   - main_idea
   - detail_extraction
   - intent
   - domain_specific
   - number_extraction only if there is a number in the transcript
8. For medical transcripts, ask about symptoms, advice, treatment, diagnosis, or health information if available.
9. For sales transcripts, ask about customer need, product/service, price, offer, request, or buying/selling information if available.
10. For general transcripts, ask about the main idea, action, reason, or important detail.

Return exactly this JSON structure and nothing else:
{{
  "qa_pairs": [
    {{
      "question": "Arabic question",
      "answer": "exact answer span copied from transcript",
      "question_type": "main_idea/detail_extraction/intent/domain_specific/number_extraction",
      "difficulty": "easy/medium"
    }}
  ]
}}
'''


def extract_json_from_text(text):
    text = str(text).strip()

    if "```json" in text:
        text = text.split("```json", 1)[1].split("```", 1)[0].strip()
    elif "```" in text:
        text = text.split("```", 1)[1].split("```", 1)[0].strip()

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        text = text[start:end+1]

    return json.loads(text)


In [ ]:
def call_llm_generate_qa(context, domain, speaker_name, n_pairs=3, max_retries=2):
    prompt = build_qa_generation_prompt(
        context=context,
        domain=domain,
        speaker_name=speaker_name,
        n_pairs=n_pairs
    )

    system_prompt = (
        "You are an Arabic NLP dataset creator. "
        "Return JSON only. Do not return explanations. "
        "The answer must be copied exactly from the transcript."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    chat_text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        chat_text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(local_llm.device)

    last_error = None

    for attempt in range(max_retries):
        try:
            with torch.no_grad():
                outputs = local_llm.generate(
                    **inputs,
                    max_new_tokens=512,
                    do_sample=True,
                    temperature=0.2,
                    top_p=0.9,
                    pad_token_id=llm_tokenizer.eos_token_id
                )

            generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
            generated_text = llm_tokenizer.decode(
                generated_ids,
                skip_special_tokens=True
            )

            parsed = extract_json_from_text(generated_text)

            if isinstance(parsed, dict) and "qa_pairs" in parsed:
                return parsed["qa_pairs"]

            return []

        except Exception as e:
            last_error = e
            print(f"Local LLM parsing error, retrying: {e}")
            time.sleep(1)

    print("Failed local generation:", last_error)
    return []


## 5. Checkpointed QA generation

This saves progress so Colab can stop/restart safely.

If the model fails on a row, that row is not marked as processed and can be retried later.


In [ ]:
def generate_llm_qa_dataframe_checkpointed(
    source_df,
    split_name,
    checkpoint_path,
    processed_path,
    context_column="reference_text",
    max_rows=None,
    n_pairs_per_transcript=3,
    save_every=10,
    sleep_seconds=0
):
    df = source_df.copy().reset_index(drop=True)

    if max_rows is not None:
        df = df.head(max_rows)

    checkpoint_path = Path(checkpoint_path)
    processed_path = Path(processed_path)

    if checkpoint_path.exists():
        all_rows = pd.read_csv(checkpoint_path).to_dict("records")
        print(f"Loaded existing checkpoint: {checkpoint_path} with {len(all_rows)} QA rows")
    else:
        all_rows = []

    if processed_path.exists():
        processed_df = pd.read_csv(processed_path)
        processed_indices = set(processed_df["source_row_index"].astype(int).tolist())
        print(f"Loaded processed log: {len(processed_indices)} transcript rows already processed")
    else:
    # ======
        # processed_indices = set()
        processed_indices = {
            int(row["source_row_index"])
            for row in all_rows
        }
        print(f"No processed log found; recovered {len(processed_indices)} source rows from the checkpoint")
    # =====

    for i, row in tqdm(df.iterrows(), total=len(df), desc=f"LLM QA generation: {split_name}"):
    # =====
        source_index = CHUNK_START + i

        # if i in processed_indices:
        if source_index in processed_indices:
            continue
    # =====

        context_for_generation = str(row[context_column]).strip()
        context_ref = str(row["reference_text"]).strip()
        context_asr = str(row["asr_text"]).strip()

        if len(context_for_generation.split()) < 5:
            print(f"Skipping short row {i}")

            # =====
            # processed_indices.add(i)
            processed_indices.add(source_index)
            # =====

            continue

        domain = row["domain"]
        speaker_name = row["speaker_name"]
        audio_file = row["audio_file"]

        qa_pairs = call_llm_generate_qa(
            context=context_for_generation,
            domain=domain,
            speaker_name=speaker_name,
            n_pairs=n_pairs_per_transcript
        )

        if not qa_pairs:
            print(f"No valid generation for row {i}. It will be retried next run.")
            continue

        added_for_this_row = 0

        for j, qa in enumerate(qa_pairs):
            if not isinstance(qa, dict):
                continue

            answer = normalize_spaces(qa.get("answer", ""))
            question = normalize_spaces(qa.get("question", ""))

            # Check answer against the context used for generation.
            answer_start_generation_context = find_answer_start_soft(context_for_generation, answer)

            # Also check answer against reference and ASR contexts.
            answer_start_ref = find_answer_start_soft(context_ref, answer)
            answer_start_asr = find_answer_start_soft(context_asr, answer)

            if not question or not answer:
                continue

            all_rows.append({
                # =====
                # "id": f"{split_name}_{CHUNK_START + i}_{j}",
                "id": f"{split_name}_{source_index}_{j}",
                # =====

                "split": split_name,

                # =====
                # "source_row_index": CHUNK_START + i,
                "source_row_index": source_index,
                # =====

                "audio_file": audio_file,
                "speaker_name": speaker_name,
                "domain": domain,

                "context_ref": context_ref,
                "context_asr": context_asr,
                "generation_context_column": context_column,

                "question": question,
                "answer": answer,

                "answer_start_generation_context": answer_start_generation_context,
                "answer_start_ref": answer_start_ref,
                "answer_start_asr": answer_start_asr,

                "valid_answer_span_generation_context": answer_start_generation_context != -1,
                "valid_answer_span_ref": answer_start_ref != -1,
                "valid_answer_span_asr": answer_start_asr != -1,

                "question_type": qa.get("question_type", ""),
                "difficulty": qa.get("difficulty", ""),

                "asr_wer": row["wer"],
                "asr_cer": row["cer"],

                "needs_manual_review": True
            })

            added_for_this_row += 1

        if added_for_this_row == 0:
            print(f"Generated output for row {i}, but no usable QA rows. It will be retried next run.")
            continue

        # =====
        # processed_indices.add(i)
        processed_indices.add(source_index)
        # =====

        if len(processed_indices) % save_every == 0:
            pd.DataFrame(all_rows).to_csv(checkpoint_path, index=False, encoding="utf-8-sig")

            # =====
            if not all_rows:
                raise RuntimeError(
                    "No QA pairs are available for this chunk. "
                    "Check generation errors and chunk contents before filtering."
                )
            # =====

            pd.DataFrame({"source_row_index": sorted(list(processed_indices))}).to_csv(
                processed_path,
                index=False,
                encoding="utf-8-sig"
            )
            print(f"Saved checkpoint for {split_name}: {len(processed_indices)} transcript rows processed")

        time.sleep(sleep_seconds)

    pd.DataFrame(all_rows).to_csv(checkpoint_path, index=False, encoding="utf-8-sig")
    pd.DataFrame({"source_row_index": sorted(list(processed_indices))}).to_csv(
        processed_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Finished {split_name}: {len(processed_indices)} transcript rows processed")
    return pd.DataFrame(all_rows)


## 6. Run QA generation for this chunk

For full test on one machine:

- `CHUNK_START = 0`
- `CHUNK_END = 950`
- `RUN_NAME = "test_full"`

For parallel work, change the chunk settings in Section 2.


In [ ]:
N_PAIRS_PER_TRANSCRIPT = 3
SAVE_EVERY = 10
SLEEP_SECONDS = 0

qa_test_df = generate_llm_qa_dataframe_checkpointed(
    test_source,
    split_name="test",

    # =====
    # checkpoint_path=f"checkpoint_llm_qa_{RUN_NAME}.csv",
    checkpoint_path=(CHECKPOINT_DIR / f"checkpoint_llm_qa_{RUN_NAME}.csv"),

    # processed_path=f"processed_llm_qa_{RUN_NAME}.csv",
    processed_path=(CHECKPOINT_DIR / f"processed_llm_qa_{RUN_NAME}.csv"),
    # =====

    context_column=GENERATE_FROM_COLUMN,
    max_rows=None,
    n_pairs_per_transcript=N_PAIRS_PER_TRANSCRIPT,
    save_every=SAVE_EVERY,
    sleep_seconds=SLEEP_SECONDS
)

print("Raw generated QA shape:", qa_test_df.shape)
display(qa_test_df.head())


## 7. Filter generated QA pairs

For final methodology, if you generated from `reference_text`, keep rows where the answer exists in `context_ref`.

For ASR evaluation later, some answers may not exist in `context_asr` because Whisper made mistakes. Do **not** remove these rows from the reference test set; they are useful for measuring ASR-caused QA drop.


In [ ]:
def filter_valid_qa_for_reference(df):
    before = len(df)
    df = df.copy()

    df["question"] = df["question"].astype(str).map(normalize_spaces)
    df["answer"] = df["answer"].astype(str).map(normalize_spaces)
    df["context_ref"] = df["context_ref"].astype(str).map(normalize_spaces)
    df["context_asr"] = df["context_asr"].astype(str).map(normalize_spaces)

    df = df[df["question"].str.len() > 5]
    df = df[df["answer"].str.len() > 0]
    df = df[df["valid_answer_span_ref"] == True]
    df = df.drop_duplicates(subset=["audio_file", "question", "answer"]).reset_index(drop=True)

    print(f"Kept {len(df)} / {before} QA rows after reference-answer filtering")
    return df


qa_test_filtered_df = filter_valid_qa_for_reference(qa_test_df)

# =====
# output_file = f"niletts_llm_qa_{RUN_NAME}_reference_vs_asr.csv"
output_file = (OUTPUT_DIR / f"niletts_llm_qa_{RUN_NAME}_reference_vs_asr.csv")
# =====

qa_test_filtered_df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Saved:", output_file)
print("Filtered shape:", qa_test_filtered_df.shape)

display(qa_test_filtered_df.head())


## 8. Quality checks

Use these checks before sending the generated file to the teammate training AraT5.


In [ ]:
print("Question type distribution:")
display(qa_test_filtered_df["question_type"].value_counts(dropna=False))

print("Difficulty distribution:")
display(qa_test_filtered_df["difficulty"].value_counts(dropna=False))

print("Valid in reference context:", qa_test_filtered_df["valid_answer_span_ref"].mean())
print("Valid in ASR context:", qa_test_filtered_df["valid_answer_span_asr"].mean())

print("Average ASR WER of generated QA rows:", qa_test_filtered_df["asr_wer"].mean())
print("Average ASR CER of generated QA rows:", qa_test_filtered_df["asr_cer"].mean())

sample_cols = [
    "audio_file", "domain", "asr_wer", "question", "answer",
    "context_ref", "context_asr", "valid_answer_span_asr"
]

display(qa_test_filtered_df[sample_cols].sample(
    min(10, len(qa_test_filtered_df)),
    random_state=SEED
))


## 9. Combine chunks later

Run this only after multiple teammates finish their chunk CSVs.

Example files:

- `niletts_llm_qa_test_part_1_reference_vs_asr.csv`
- `niletts_llm_qa_test_part_2_reference_vs_asr.csv`


In [ ]:
# import glob

# # =====
# # chunk_files = sorted(glob.glob("niletts_llm_qa_test_part_*_reference_vs_asr.csv"))
# chunk_files = sorted(OUTPUT_DIR.glob("niletts_llm_qa_test_part_*_reference_vs_asr.csv"))
# # =====

# print("Chunk files:")
# for f in chunk_files:
#     print(f)

# combined_df = pd.concat([pd.read_csv(f) for f in chunk_files], ignore_index=True)
# combined_df = combined_df.drop_duplicates(subset=["audio_file", "question", "answer"]).reset_index(drop=True)

# # =====
# # combined_df.to_csv(
# #     "niletts_llm_qa_test_reference_vs_asr_combined.csv",
# #     index=False,
# #     encoding="utf-8-sig"
# # )

# combined_df.to_csv(
#     OUTPUT_DIR / "niletts_llm_qa_test_reference_vs_asr_combined.csv",
#     index=False,
#     encoding="utf-8-sig"
# )
# # =====

# print("Combined shape:", combined_df.shape)
# display(combined_df.head())


## 10. Free GPU memory after generation

Run this before training AraT5 in another notebook.


In [ ]:
try:
    del local_llm
    del llm_tokenizer
except NameError:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA memory cleared.")
